# 📚 템플릿 4 — 강의 자료 PDF 요약 + Q&A

PDF 업로드 → 전체 요약 + RAG로 질문 답변. 7회차 RAG 응용입니다.

## 핵심 흐름
1. PDF 파싱 → 텍스트 추출
2. 청크 단위로 임베딩 → ChromaDB
3. 요약 생성 (LLM)
4. 질문 → 관련 청크 검색 → LLM 답변

## 학생이 가장 쉽게 손댈 곳
- 청크 크기 (`chunk_size`)
- 요약 프롬프트 스타일 ("3줄로", "초등학생 수준", "시험 대비")
- UI 디자인

In [ ]:
!pip install -q gradio openai pypdf sentence-transformers chromadb

In [ ]:
# ===== 서버 연결 =====
SERVER_URL = "https://YOUR_URL.trycloudflare.com".strip().rstrip("/")
MODEL = "qwen2.5:7b-instruct"
assert "YOUR" not in SERVER_URL, "❌ SERVER_URL을 강사가 알려준 URL로 바꾸세요"

import httpx
from openai import OpenAI
client = OpenAI(base_url=f"{SERVER_URL}/v1", api_key="ollama",
                http_client=httpx.Client(headers={"User-Agent": "Mozilla/5.0"}))

# 임베딩 모델
print("임베딩 모델 로딩 중...")
import chromadb
from chromadb.utils import embedding_functions
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-m3")
chroma_client = chromadb.Client()
print("✅ 준비 완료")

In [ ]:
import gradio as gr
import pypdf

state = {"kb": None, "summary": "", "title": ""}

def chunk_text(text, size=500, overlap=80):
    chunks = []
    i = 0
    while i < len(text):
        chunks.append(text[i:i+size])
        i += size - overlap
    return chunks

def load_pdf(pdf_file, summary_style):
    if pdf_file is None:
        return "PDF를 업로드해주세요", []
    
    reader = pypdf.PdfReader(pdf_file.name)
    full_text = "\n".join(p.extract_text() or "" for p in reader.pages)
    
    if len(full_text) < 100:
        return "PDF에서 텍스트를 추출하지 못했습니다 (이미지 PDF인가요?)", []
    
    # 청크 + KB 구축
    chunks = chunk_text(full_text)
    try: chroma_client.delete_collection("user_pdf")
    except: pass
    kb = chroma_client.create_collection("user_pdf", embedding_function=ef)
    kb.add(documents=chunks, ids=[f"c{i}" for i in range(len(chunks))])
    state["kb"] = kb
    state["title"] = pdf_file.name.split("/")[-1]
    
    # 요약 (스타일별 프롬프트)
    style_prompts = {
        "📝 깔끔 요약 (불릿 5개)": "핵심을 5개 불릿으로 깔끔하게",
        "🎓 시험 대비 (핵심 개념)": "시험에 나올 핵심 개념을 정리해서",
        "🧒 초등학생 설명": "초등학생이 이해할 정도로 쉽게 비유하면서",
        "📊 구조 정리 (계층화)": "주제 > 소주제 > 세부내용 계층 구조로",
    }
    style_inst = style_prompts.get(summary_style, "핵심을 정리해서")
    
    summary_prompt = f"""다음 PDF 내용을 한국어로 {style_inst} 요약해주세요:

{full_text[:5000]}

[요약]"""
    
    summary = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": summary_prompt}],
        max_tokens=600, temperature=0.3,
    ).choices[0].message.content
    state["summary"] = summary
    
    info = f"## 📄 {state['title']}\n- 페이지: **{len(reader.pages)}**\n- 청크: **{len(chunks)}**\n\n## 📝 요약\n\n{summary}"
    return info, []

def ask(question, history):
    if state["kb"] is None:
        return history + [(question, "PDF를 먼저 업로드해주세요")]
    
    res = state["kb"].query(query_texts=[question], n_results=3)
    ctx = "\n\n".join(f"[자료 {i+1}]\n{c}" for i, c in enumerate(res["documents"][0]))
    
    prompt = f"""다음 자료를 근거로 질문에 답해주세요. 
자료에 없는 내용은 "자료에 명시되어 있지 않습니다"라고 답하세요.
간결하게 3-5문장으로.

[자료]
{ctx}

[질문] {question}

[답변]"""
    
    answer = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=400, temperature=0.2,
    ).choices[0].message.content
    
    return history + [(question, answer)]

with gr.Blocks(title="📚 PDF 학습 도우미", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📚 강의 자료 PDF 요약 + Q&A")
    with gr.Row():
        with gr.Column(scale=1):
            pdf = gr.File(label="📄 PDF 업로드", file_types=[".pdf"])
            style = gr.Dropdown(
                ["📝 깔끔 요약 (불릿 5개)", "🎓 시험 대비 (핵심 개념)",
                 "🧒 초등학생 설명", "📊 구조 정리 (계층화)"],
                value="📝 깔끔 요약 (불릿 5개)",
                label="🎨 요약 스타일",
            )
            load_btn = gr.Button("➤ 분석 시작", variant="primary", size="lg")
            info = gr.Markdown("PDF를 업로드하고 버튼을 눌러주세요")
        with gr.Column(scale=2):
            chat = gr.Chatbot(label="💬 Q&A", height=500)
            msg = gr.Textbox(label="질문", placeholder="예: 이 챕터의 핵심 3가지는?")
            ask_btn = gr.Button("➤ 질문하기")
    
    load_btn.click(load_pdf, [pdf, style], [info, chat])
    ask_btn.click(ask, [msg, chat], chat).then(lambda: "", outputs=msg)
    msg.submit(ask, [msg, chat], chat).then(lambda: "", outputs=msg)

demo.launch(share=True)

---
## 🚀 바이브 코딩 확장 아이디어

### 쉬움
- 요약 스타일 추가 (4컷 만화 줄거리, 한 줄 요약, FAQ 형태)
- 답변 글자 수 슬라이더로 조정
- 출처 표시 (어느 페이지/청크에서 가져왔는지)

### 중간
- 여러 PDF 동시 업로드해서 cross-document Q&A
- 자동 퀴즈 모드: PDF 내용으로 5문제 출제 → 학생 풀기
- 강의 자료 + 본인 노트 합쳐서 검색
- 음성 질문 (Whisper STT 붙이기)

### 도전적
- 수식 OCR (이미지 PDF에서 수식 추출 → LaTeX)
- 강의 슬라이드별 자동 카드뉴스 생성
- 다국어 PDF 자동 번역 후 한국어 Q&A
- 친구들끼리 같은 PDF로 공동 노트 작성

### 🎁 자랑하기 팁
- 시험 직전 동기들한테 "내 PDF 요약 봇" URL 공유 → 인기 폭발
- 본인 학과 전공 PDF로 만들면 후배들한테 전수 가능